# Stage sample ancestry TSVs for SV discovery

Builds `sample_id \t ancestry` tables for `AnnotateSvCallset.sample_ancestry_tsv`.
Discovery plots order samples non-African first, then African
(`sv_annotation/docs/definitions.md`).

**IDs** must match VCF sample names. Default source is
`covariates.source_rebuilt.csv.gz` (`research_id` + `ancestry_pred`).

**Ancestry labels** written to the TSV: `eas`, `amr`, `eur`, `sas`, `afr`, `oth`.
`mid` and missing predictions map to `oth`.

**Phase membership** (if you do not set a VCF URI):
- Phase 1: `lr_phase` in `phase_1`, `phase_1_phase_2`
- Phase 2: `lr_phase` in `phase_2`, `phase_1_phase_2`

If you set `PHASE1_MAIN_VCF` / `PHASE2_MAIN_VCF`, the notebook lists samples from
that VCF header and keeps only those IDs (preferred).

Run this **before** submitting `AnnotateSvCallset`. `sv_03_manuscript_stats.ipynb`
is post-workflow.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Bootstrap scripts/ from $WORKSPACE_BUCKET/scripts/ when not on the VM.
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook

SCRIPTS = init_notebook("workspace_paths.py", "sv_site_utils.py")
from workspace_paths import data_root

import gzip
import shutil
import subprocess

import pandas as pd
from sv_site_utils import order_samples_by_ancestry  # noqa: E402

ROOT = data_root()

WORK = Path(os.environ.get("SAMPLE_ANCESTRY_WORK", Path.cwd() / "sample_ancestry_work")).resolve()
WORK.mkdir(parents=True, exist_ok=True)

WORKSPACE_BUCKET = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
GCS_PREFIX = os.environ.get(
    "SAMPLE_ANCESTRY_GCS_PREFIX",
    f"{WORKSPACE_BUCKET}/metadata" if WORKSPACE_BUCKET else "",
)
COV_GCS = os.environ.get(
    "TRACTOR_COVARIATES_GCS",
    f"{WORKSPACE_BUCKET}/covariates/covariates.source_rebuilt.csv.gz"
    if WORKSPACE_BUCKET
    else "",
)
COV_CSV = Path(os.environ.get("SAMPLE_ANCESTRY_COV_CSV", ROOT / "covariates.source_rebuilt.csv.gz"))

PHASE1_MAIN_VCF = os.environ.get("SV_PHASE1_MAIN_VCF", "").strip()
PHASE2_MAIN_VCF = os.environ.get("SV_PHASE2_MAIN_VCF", "").strip()

CANONICAL = {"eas", "amr", "eur", "sas", "afr", "oth"}
TO_OTH = {"", "nan", "none", "unknown", "other", "mid", "oth"}

print("ROOT:", ROOT)
print("COV_CSV:", COV_CSV)
print("COV_GCS:", COV_GCS or "(unset)")
print("PHASE1_MAIN_VCF:", PHASE1_MAIN_VCF or "(lr_phase filter)")
print("PHASE2_MAIN_VCF:", PHASE2_MAIN_VCF or "(lr_phase filter)")
print("GCS_PREFIX:", GCS_PREFIX or "(set WORKSPACE_BUCKET or SAMPLE_ANCESTRY_GCS_PREFIX)")


In [ ]:
def run(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("+", " ".join(map(str, cmd)))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout, end="" if proc.stdout.endswith("\n") else "\n")
    if proc.stderr:
        print(proc.stderr, end="" if proc.stderr.endswith("\n") else "\n")
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(
            proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr
        )
    return proc


def canonicalize_ancestry(value) -> str:
    lab = "" if pd.isna(value) else str(value).strip().lower()
    if lab in CANONICAL:
        return lab
    if lab in TO_OTH:
        return "oth"
    if lab in {"african", "afr"}:
        return "afr"
    return "oth"


def list_vcf_samples(uri: str) -> list[str]:
    """Read sample IDs from a VCF/BCF header. Streams gs:// via gsutil cat."""
    if shutil.which("bcftools") and not uri.startswith("gs://"):
        proc = run(["bcftools", "query", "-l", uri])
        return [s for s in proc.stdout.splitlines() if s]

    if uri.startswith("gs://"):
        proc = subprocess.Popen(["gsutil", "cat", uri], stdout=subprocess.PIPE)
        assert proc.stdout is not None
        fh = gzip.open(proc.stdout, "rt") if uri.endswith(".gz") else proc.stdout
    elif uri.endswith(".gz"):
        proc = None
        fh = gzip.open(uri, "rt")
    else:
        proc = None
        fh = open(uri, "rt")

    try:
        for line in fh:
            if line.startswith("#CHROM"):
                return line.rstrip("\n").split("\t")[9:]
        raise RuntimeError(f"No #CHROM header line in {uri}")
    finally:
        fh.close()
        if proc is not None:
            proc.terminate()
            proc.wait()

In [ ]:
if not COV_CSV.exists() and COV_GCS:
    print(f"Downloading {COV_GCS}")
    run(["gsutil", "cp", COV_GCS, str(COV_CSV)])

assert COV_CSV.exists(), (
    f"Missing {COV_CSV}. Set TRACTOR_COVARIATES_GCS or copy covariates.source_rebuilt.csv.gz "
    "next to the notebook / under tractor_mix/."
)

cov = pd.read_csv(
    COV_CSV,
    dtype={"research_id": str, "biobank_id": str},
    low_memory=False,
)
assert cov["research_id"].is_unique, "research_id must be unique"
cov["sample_id"] = cov["research_id"].astype(str).str.strip()
pred = cov["ancestry_pred"] if "ancestry_pred" in cov.columns else pd.Series(pd.NA, index=cov.index)
other = (
    cov["ancestry_pred_other"]
    if "ancestry_pred_other" in cov.columns
    else pd.Series(pd.NA, index=cov.index)
)
cov["ancestry"] = pred.map(canonicalize_ancestry)
missing_pred = pred.isna() | (pred.astype(str).str.strip() == "")
cov.loc[missing_pred, "ancestry"] = other[missing_pred].map(canonicalize_ancestry)
print("covariates:", len(cov))
print(cov["lr_phase"].value_counts(dropna=False).to_string())
print("\nancestry after canonicalize:")
print(cov["ancestry"].value_counts(dropna=False).to_string())

In [ ]:
PHASE_FILTERS = {
    "phase1": {"phase_1", "phase_1_phase_2"},
    "phase2": {"phase_2", "phase_1_phase_2"},
}
VCF_BY_PHASE = {"phase1": PHASE1_MAIN_VCF, "phase2": PHASE2_MAIN_VCF}

id_to_anc = dict(zip(cov["sample_id"], cov["ancestry"]))
if "biobank_id" in cov.columns:
    for rid, bid, anc in zip(cov["sample_id"], cov["biobank_id"], cov["ancestry"]):
        if pd.notna(bid) and str(bid).strip():
            id_to_anc.setdefault(str(bid).strip(), anc)

outputs: dict[str, Path] = {}
summaries: list[dict] = []

for phase, labels in PHASE_FILTERS.items():
    vcf_uri = VCF_BY_PHASE[phase]
    if vcf_uri:
        samples = list_vcf_samples(vcf_uri)
        source = f"vcf:{vcf_uri}"
    else:
        samples = cov.loc[cov["lr_phase"].isin(labels), "sample_id"].tolist()
        source = f"lr_phase in {sorted(labels)}"
        print(
            f"WARNING: {phase} has no main VCF URI; using {source}. "
            "Set SV_PHASE1_MAIN_VCF / SV_PHASE2_MAIN_VCF so IDs match the callset."
        )

    rows = []
    n_unmatched = 0
    for sample_id in samples:
        if sample_id in id_to_anc:
            anc = id_to_anc[sample_id]
        else:
            anc = "oth"
            n_unmatched += 1
        rows.append({"sample_id": sample_id, "ancestry": anc})

    table = pd.DataFrame(rows).drop_duplicates(subset=["sample_id"], keep="first")
    order = order_samples_by_ancestry(dict(zip(table["sample_id"], table["ancestry"])))
    table = table.set_index("sample_id").loc[order].reset_index()

    out = WORK / f"{phase}.sample_ancestry.tsv"
    table.to_csv(out, sep="\t", index=False)
    outputs[phase] = out
    summaries.append(
        {
            "phase": phase,
            "n_samples": int(len(table)),
            "n_unmatched_to_covariates": n_unmatched,
            "source": source,
            "ancestry_counts": table["ancestry"].value_counts().to_dict(),
            "first_sample": table["sample_id"].iloc[0] if len(table) else None,
            "first_ancestry": table["ancestry"].iloc[0] if len(table) else None,
        }
    )
    print(f"\n{phase}: {len(table):,} samples  unmatched={n_unmatched}  source={source}")
    print(table["ancestry"].value_counts().to_string())
    print("wrote", out)

MANIFEST = WORK / "sample_ancestry.manifest.json"
MANIFEST.write_text(
    json.dumps(
        {
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "covariates": str(COV_CSV),
            "summaries": summaries,
        },
        indent=2,
    )
    + "\n"
)
print("\nmanifest:", MANIFEST)
for summary in summaries:
    counts = summary["ancestry_counts"]
    if any(label != "afr" and n for label, n in counts.items()):
        assert summary["first_ancestry"] != "afr", (
            f"{summary['phase']}: discovery order should start with a non-African sample."
        )

In [ ]:
if not GCS_PREFIX:
    raise SystemExit(
        "Set WORKSPACE_BUCKET (Terra default) or SAMPLE_ANCESTRY_GCS_PREFIX before uploading."
    )

wdl_lines = []
for phase, path in outputs.items():
    dest = f"{GCS_PREFIX}/{path.name}"
    print(f"Uploading {path} -> {dest}")
    run(["gsutil", "cp", str(path), dest])
    wdl_lines.append((phase, dest))

run(["gsutil", "cp", str(MANIFEST), f"{GCS_PREFIX}/sample_ancestry.manifest.json"])
run(["gsutil", "ls", "-lh", f"{GCS_PREFIX}/"])

print("\nUse these WDL inputs:")
for phase, uri in wdl_lines:
    print(f'  "AnnotateSvCallset.sample_ancestry_tsv": "{uri}",  # {phase}')

## Optional cleanup

In [ ]:
# shutil.rmtree(WORK)
# print("Removed", WORK)